# Regression Analysis

This notebook uses the previously cleaned datasets for gas and electricity usage and provides a regression analysis of the data.

Available DataFrames:
- `gas_monthly_usage`
- `electric_monthly_usage`

This version keeps more of the original wording and narrative, while cleaning the code, improving the organization, and making the regression comparisons easier to follow.


### Accessing the dataset

First, equivalently to how data was fetched in the EDA notebook, fetch the data for gas usage and electricity usage and save the data as dataframes. Also do a visual check of the imported data.


In [1]:

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "clean").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing data/clean.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "clean"

gas_monthly_usage = pd.read_csv(DATA_DIR / "gas_monthly_usage.csv")
electric_monthly_usage = pd.read_csv(DATA_DIR / "electric_monthly_usage.csv")

print("Gas dataset shape:", gas_monthly_usage.shape)
print("Electricity dataset shape:", electric_monthly_usage.shape)

gas_monthly_usage.head()


Gas dataset shape: (8412, 6)
Electricity dataset shape: (19656, 6)


,Property Name,Portfolio Manager ID,Property Type - Self-Selected,Gross Floor Area,month,normalized_monthly_usage
0,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-01-01,23839.31
1,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-02-01,18735.80
2,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-03-01,14082.81
3,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-04-01,11537.50
4,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-05-01,7784.48


### Additional Data cleaning

- Filter the dataframes to include only pertinent columns (features and the output variable)
- Drop rows with placeholder values for the gross floor area (i.e. where GFA = 1)
- Keep the month variable in a format that can be treated cleanly as a categorical variable later


In [2]:

gas_usage_for_regression = gas_monthly_usage[[
    "Gross Floor Area",
    "month",
    "Property Type - Self-Selected",
    "normalized_monthly_usage",
]].copy()

electricity_usage_for_regression = electric_monthly_usage[[
    "Gross Floor Area",
    "month",
    "Property Type - Self-Selected",
    "normalized_monthly_usage",
]].copy()

for name, df in {
    "gas": gas_usage_for_regression,
    "electricity": electricity_usage_for_regression,
}.items():
    df["month"] = pd.to_datetime(df["month"])
    df.dropna(
        subset=[
            "Gross Floor Area",
            "month",
            "Property Type - Self-Selected",
            "normalized_monthly_usage",
        ],
        inplace=True,
    )
    df = df[df["Gross Floor Area"] != 1]

    if name == "gas":
        gas_usage_for_regression = df.copy()
    else:
        electricity_usage_for_regression = df.copy()

print("Gas regression dataframe shape:", gas_usage_for_regression.shape)
print("Electricity regression dataframe shape:", electricity_usage_for_regression.shape)


Gas regression dataframe shape: (8052, 4)
Electricity regression dataframe shape: (13692, 4)


### Quick checks

Before running the models, do a quick check of the regression datasets to confirm that the property types and the basic GFA vs usage patterns look reasonable.


In [ ]:

print("Gas property types:")
print(gas_usage_for_regression["Property Type - Self-Selected"].value_counts())

print("Electricity property types:")
print(electricity_usage_for_regression["Property Type - Self-Selected"].value_counts())


SyntaxError: unterminated string literal (detected at line 4) (176408049.py, line 4)

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    gas_usage_for_regression["Gross Floor Area"],
    gas_usage_for_regression["normalized_monthly_usage"],
    alpha=0.5,
)
ax.set_title("Gas Usage vs Gross Floor Area")
ax.set_xlabel("Gross Floor Area")
ax.set_ylabel("Normalized Monthly Usage")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    electricity_usage_for_regression["Gross Floor Area"],
    electricity_usage_for_regression["normalized_monthly_usage"],
    alpha=0.5,
)
ax.set_title("Electricity Usage vs Gross Floor Area")
ax.set_xlabel("Gross Floor Area")
ax.set_ylabel("Normalized Monthly Usage")
plt.show()


### Simple regression analysis only using GFA and monthly usage

First we conduct a simple regression analysis, utilizing the GFA (feature) and normalized monthly usage (output) variables alone. Based on the scatterplots from the EDA notebook, we expect this analysis to be somewhat unfulfilling, but it provides a useful baseline for later comparisons.


In [ ]:

def fit_and_evaluate_linear_model(X, y, test_size=0.15, random_state=28):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return {
        "model": model,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "y_pred": y_pred,
        "mse": mse,
        "r2": r2,
    }


def prepare_base_features(df):
    X = df[["Gross Floor Area", "Property Type - Self-Selected", "month"]].copy()
    X["month"] = pd.to_datetime(X["month"]).dt.month_name()

    X_encoded = pd.get_dummies(
        X,
        columns=["Property Type - Self-Selected", "month"],
        drop_first=True,
        dtype=float,
    )
    return X_encoded


def add_gfa_property_type_interactions(X_encoded):
    X_interaction = X_encoded.copy()
    property_cols = [
        col for col in X_interaction.columns
        if col.startswith("Property Type - Self-Selected_")
    ]

    for col in property_cols:
        X_interaction[f"GFA_x_{col}"] = (
            X_interaction["Gross Floor Area"] * X_interaction[col]
        )

    return X_interaction


def model_summary_dataframe(feature_names, coefficients):
    return pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": coefficients,
    }).sort_values(by="Coefficient", key=abs, ascending=False)



## 5. Simple regression using GFA only

This model uses only Gross Floor Area as the predictor. It serves as a baseline for comparison.


In [ ]:

X_gas_simple = gas_usage_for_regression[["Gross Floor Area"]]
y_gas = gas_usage_for_regression["normalized_monthly_usage"]

gas_simple_results = fit_and_evaluate_linear_model(X_gas_simple, y_gas)

print("Gas simple regression")
print("MSE:", gas_simple_results["mse"])
print("R2:", gas_simple_results["r2"])


In [ ]:

X_electric_simple = electricity_usage_for_regression[["Gross Floor Area"]]
y_electric = electricity_usage_for_regression["normalized_monthly_usage"]

electric_simple_results = fit_and_evaluate_linear_model(X_electric_simple, y_electric)

print("Electricity simple regression")
print("MSE:", electric_simple_results["mse"])
print("R2:", electric_simple_results["r2"])


Simple visual checks

A plot of actual vs predicted values helps show whether the simple regression is tracking the data well. If the points stray far from the diagonal, then GFA alone is not capturing enough of the variation in usage.


In [ ]:

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(gas_simple_results["y_test"], gas_simple_results["y_pred"], alpha=0.6)
ax.plot(
    [gas_simple_results["y_test"].min(), gas_simple_results["y_test"].max()],
    [gas_simple_results["y_test"].min(), gas_simple_results["y_test"].max()],
)
ax.set_title("Gas Simple Regression: Actual vs Predicted")
ax.set_xlabel("Actual Usage")
ax.set_ylabel("Predicted Usage")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(electric_simple_results["y_test"], electric_simple_results["y_pred"], alpha=0.6)
ax.plot(
    [electric_simple_results["y_test"].min(), electric_simple_results["y_test"].max()],
    [electric_simple_results["y_test"].min(), electric_simple_results["y_test"].max()],
)
ax.set_title("Electricity Simple Regression: Actual vs Predicted")
ax.set_xlabel("Actual Usage")
ax.set_ylabel("Predicted Usage")
plt.show()


### Regression analysis using GFA, property type and month

We now take a step further and conduct an analysis adding the property type and month as categorical variables. This lets the model account for both differences in building category and differences across the year.


In [ ]:

X_gas_base = prepare_base_features(gas_usage_for_regression)
y_gas = gas_usage_for_regression["normalized_monthly_usage"]

gas_base_results = fit_and_evaluate_linear_model(X_gas_base, y_gas)
gas_base_coef_df = model_summary_dataframe(
    X_gas_base.columns,
    gas_base_results["model"].coef_,
)

print("Gas multivariable regression")
print("Intercept:", gas_base_results["model"].intercept_)
print("MSE:", gas_base_results["mse"])
print("R2:", gas_base_results["r2"])

gas_base_coef_df.head(20)


In [ ]:

X_electric_base = prepare_base_features(electricity_usage_for_regression)
y_electric = electricity_usage_for_regression["normalized_monthly_usage"]

electric_base_results = fit_and_evaluate_linear_model(X_electric_base, y_electric)
electric_base_coef_df = model_summary_dataframe(
    X_electric_base.columns,
    electric_base_results["model"].coef_,
)

print("Electricity multivariable regression")
print("Intercept:", electric_base_results["model"].intercept_)
print("MSE:", electric_base_results["mse"])
print("R2:", electric_base_results["r2"])

electric_base_coef_df.head(20)


Visualizing the role of Property Type

These plots are useful for checking whether some property types appear to cluster differently. This helps motivate the interaction analysis later, especially if the relationship between GFA and usage seems to vary by property type.


In [ ]:

fig, ax = plt.subplots(figsize=(8, 6))
for property_type in gas_usage_for_regression["Property Type - Self-Selected"].dropna().unique():
    subset = gas_usage_for_regression[
        gas_usage_for_regression["Property Type - Self-Selected"] == property_type
    ]
    ax.scatter(
        subset["Gross Floor Area"],
        subset["normalized_monthly_usage"],
        alpha=0.5,
        label=property_type,
    )

ax.set_title("Gas Usage by Property Type")
ax.set_xlabel("Gross Floor Area")
ax.set_ylabel("Normalized Monthly Usage")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(8, 6))
for property_type in electricity_usage_for_regression["Property Type - Self-Selected"].dropna().unique():
    subset = electricity_usage_for_regression[
        electricity_usage_for_regression["Property Type - Self-Selected"] == property_type
    ]
    ax.scatter(
        subset["Gross Floor Area"],
        subset["normalized_monthly_usage"],
        alpha=0.5,
        label=property_type,
    )

ax.set_title("Electricity Usage by Property Type")
ax.set_xlabel("Gross Floor Area")
ax.set_ylabel("Normalized Monthly Usage")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()


### Regression Using GFA, Property Type, Month, and an Interaction Term

Now we consider an interaction term in the model including all variables. We use `GFA × Property Type` as the interaction term, since there may be a relationship between building size and the type of property. This allows different property types to have different GFA slopes rather than forcing one common slope across all of them.


In [ ]:

X_gas_interaction = add_gfa_property_type_interactions(X_gas_base)
y_gas = gas_usage_for_regression["normalized_monthly_usage"]

gas_interaction_results = fit_and_evaluate_linear_model(X_gas_interaction, y_gas)
gas_interaction_coef_df = model_summary_dataframe(
    X_gas_interaction.columns,
    gas_interaction_results["model"].coef_,
)

print("Gas interaction regression")
print("Intercept:", gas_interaction_results["model"].intercept_)
print("MSE:", gas_interaction_results["mse"])
print("R2:", gas_interaction_results["r2"])

gas_interaction_coef_df.head(30)


In [ ]:

X_electric_interaction = add_gfa_property_type_interactions(X_electric_base)
y_electric = electricity_usage_for_regression["normalized_monthly_usage"]

electric_interaction_results = fit_and_evaluate_linear_model(X_electric_interaction, y_electric)
electric_interaction_coef_df = model_summary_dataframe(
    X_electric_interaction.columns,
    electric_interaction_results["model"].coef_,
)

print("Electricity interaction regression")
print("Intercept:", electric_interaction_results["model"].intercept_)
print("MSE:", electric_interaction_results["mse"])
print("R2:", electric_interaction_results["r2"])

electric_interaction_coef_df.head(30)


### Model comparison tables

These tables summarize the performance of the three model families for gas and electricity:

- Simple regression using GFA only
- A multivariable model using GFA, property type, and month
- A multivariable model including the `GFA × Property Type` interaction


In [ ]:

gas_model_comparison = pd.DataFrame({
    "Model": [
        "Simple: GFA only",
        "Base: GFA + Property Type + Month",
        "Interaction: Base + GFA × Property Type",
    ],
    "MSE": [
        gas_simple_results["mse"],
        gas_base_results["mse"],
        gas_interaction_results["mse"],
    ],
    "R2": [
        gas_simple_results["r2"],
        gas_base_results["r2"],
        gas_interaction_results["r2"],
    ],
})

gas_model_comparison


In [ ]:

electric_model_comparison = pd.DataFrame({
    "Model": [
        "Simple: GFA only",
        "Base: GFA + Property Type + Month",
        "Interaction: Base + GFA × Property Type",
    ],
    "MSE": [
        electric_simple_results["mse"],
        electric_base_results["mse"],
        electric_interaction_results["mse"],
    ],
    "R2": [
        electric_simple_results["r2"],
        electric_base_results["r2"],
        electric_interaction_results["r2"],
    ],
})

electric_model_comparison


### Interpretation notes

The comparison tables and coefficient tables can be used to support conclusions such as:

- A low R² in the simple model suggests that GFA alone does not explain enough of the variation in energy usage
- A higher R² in the multivariable model suggests that property type and month add meaningful predictive information
- A large improvement after adding `GFA × Property Type` suggests that the relationship between building size and usage differs across property types

This is especially important for electricity usage, where the interaction model may reveal that one common GFA slope is too simplistic.


### Optional next steps

Further extensions could include:

- Adding `GFA × Month` interaction terms
- Investigating outliers more formally using IQR or z-score methods
- Comparing train and test scores side by side
- Trying regularized models such as Ridge or Lasso if the interaction model becomes unstable
